# DATA620 Project 1 — Centrality by Category (v004)

**Aligned to proposal:** public network source + node category + degree centrality comparison + statistical test.

## Requirement Alignment
- Public dataset source with node-level category ✅
- Graph built from node/edge files ✅
- Degree centrality by category ✅
- Statistical comparison across categories ✅

## Dataset Source
- Zachary Karate Club (public social network dataset)
- Reference: https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html
- Local files used:
  - `data/project1_nodes-v004.csv` (`node_id`, `category`)
  - `data/project1_edges-v004.csv` (`source`, `target`)

In [1]:
import pandas as pd
import networkx as nx
from scipy.stats import ttest_ind

nodes=pd.read_csv("data/project1_nodes-v004.csv")
edges=pd.read_csv("data/project1_edges-v004.csv")
nodes.head(), edges.head()

(   node_id category
 0        0   Mr. Hi
 1        1   Mr. Hi
 2        2   Mr. Hi
 3        3   Mr. Hi
 4        4   Mr. Hi,
    source  target
 0       0       1
 1       0       2
 2       0       3
 3       0       4
 4       0       5)

In [2]:
G=nx.from_pandas_edgelist(edges, source="source", target="target")
cat_map=dict(zip(nodes.node_id, nodes.category))
nx.set_node_attributes(G, cat_map, "category")

deg=nx.degree_centrality(G)
rows=[]
for n in G.nodes():
    rows.append({"node_id":n,"category":cat_map.get(n,"Unknown"),"degree_centrality":deg[n]})

df=pd.DataFrame(rows)
df.sort_values("degree_centrality", ascending=False).head(10)

,node_id,category,degree_centrality
23,33,Officer,0.515152
0,0,Mr. Hi,0.484848
21,32,Officer,0.363636
2,2,Mr. Hi,0.303030
1,1,Mr. Hi,0.272727
3,3,Mr. Hi,0.181818
16,31,Officer,0.181818
29,23,Officer,0.151515
12,13,Mr. Hi,0.151515
8,8,Mr. Hi,0.151515


In [3]:
summary=df.groupby("category")["degree_centrality"].agg(["mean","median","std","count"])
summary

,mean,median,std,count
category,,,,
Mr. Hi,0.144385,0.121212,0.114657,17
Officer,0.133690,0.090909,0.123584,17


In [4]:
groups=list(df.groupby("category"))
a=groups[0][1]["degree_centrality"]
b=groups[1][1]["degree_centrality"]
res=ttest_ind(a,b,equal_var=False)
res

TtestResult(statistic=np.float64(0.26158135253004317), pvalue=np.float64(0.7953299431961781), df=np.float64(31.821767232451297))

## Interpretation
- Category-level centrality differences were tested using Welch t-test (two-category case).
- This follows the proposal workflow: category-aware centrality analysis with a reproducible public dataset.